In [ ]:
file_path = '/content/drive/MyDrive/FYP/' # define the file path to save the datasets

In [ ]:
# Importing necessary libraries
import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer

# Dataset 1

## Download dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("suchintikasarkar/sentiment-analysis-for-mental-health")

small_datafile = path + '/Combined Data.csv'

df_small = pd.read_csv(small_datafile, encoding="utf-8", encoding_errors='replace')

print("\nDataset from Kaggle:")
print(df_small.head(3))

100%|██████████| 11.1M/11.1M [00:00<00:00, 91.7MB/s]

Extracting files...



Dataset from Kaggle:
   Unnamed: 0                                          statement   status
0           0                                         oh my gosh  Anxiety
1           1  trouble sleeping, confused mind, restless hear...  Anxiety
2           2  All wrong, back off dear, forward doubt. Stay ...  Anxiety


## Remove unnecessary index column

In [ ]:
df_small = df_small.drop(columns ='Unnamed: 0')
print(df_small.head(3))

                                           statement   status
0                                         oh my gosh  Anxiety
1  trouble sleeping, confused mind, restless hear...  Anxiety
2  All wrong, back off dear, forward doubt. Stay ...  Anxiety


## Check for null values

In [ ]:
has_null_statement = df_small['statement'].isnull().any()
has_null_status = df_small['status'].isnull().any()
print("Does statement have null values?: " + str(has_null_statement))
print("Does status have null values?   : " + str(has_null_status))

Does statement have null values?: True
Does status have null values?   : False


## Remove null values

In [ ]:
df_small = df_small.dropna(axis=0, ignore_index=True)

print("After null value removal: ")
print(df_small.isnull().value_counts())
print("\nRemaining instances: " + str(len(df_small)))

After null value removal: 
statement  status
False      False     52681
Name: count, dtype: int64

Remaining instances: 52681


## Remove duplicates

In [ ]:
df_small = df_small.drop_duplicates(ignore_index=True)
print("Remaining instances: " + str(len(df_small)))

Remaining instances: 51093


In [ ]:
df_small['status'].value_counts()

,count
status,
Normal,16040
Depression,15094
Suicidal,10644
Anxiety,3623
Bipolar,2501
Stress,2296
Personality disorder,895


## Combine instances of the same text but with different status using multilabel binarizer.

In [ ]:
df_small.loc[df_small.duplicated(subset='statement', keep=False)].head(2)

,statement,status
8042,"All this work, all this pressure that everyone...",Suicidal
8043,"All this work, all this pressure that everyone...",Depression


In [ ]:
def get_multilabels(df, label:str, text:str):
    grouped = df.groupby(text)[label].apply(list).reset_index()

    mlb = MultiLabelBinarizer()

    label_matrix = mlb.fit_transform(grouped[label])

    df_label_matrix = pd.DataFrame(
        label_matrix,
        columns=mlb.classes_,
        index=grouped.index
    )

    return pd.concat([grouped[text], df_label_matrix], axis=1)

df_small_multilabel = get_multilabels(df_small, "status", "statement")
df_small_multilabel.head(3)

,statement,Anxiety,Bipolar,Depression,Normal,Personality disorder,Stress,Suicidal
0,"\tIt was the summer, I had just started a new ...",0,0,0,1,0,0,0
1,...,0,0,0,0,0,1,0
2,"He grew from a short, stubby, orange hair...",0,0,0,1,0,0,0


In [ ]:
test_statement = df_small.loc[df_small.duplicated(subset='statement', keep=False)].iat[0,0]
df_small.loc[df_small['statement'] == (test_statement)]

# the following statement should have a 1 in Depression and Suicidal

,statement,status
8042,"All this work, all this pressure that everyone...",Suicidal
8043,"All this work, all this pressure that everyone...",Depression


In [ ]:
# checking for any duplicates in the duplicates in the statement column
df_small_multilabel.duplicated(subset='statement', keep=False).any()

np.False_

In [ ]:
rows_all_zero = df_small_multilabel[(df_small_multilabel[['Anxiety', 'Bipolar', 'Depression', 'Normal','Personality disorder', 'Stress', 'Suicidal']] == 0).all(axis=1)]
print("Numnber of rows with all zeroes: " + str(rows_all_zero.size))

Numnber of rows with all zeroes: 0


In [ ]:
len(df_small_multilabel)

51073

In [ ]:
df_small = df_small_multilabel.copy()
df_small_multilabel = [] # remove df_small_multilabel from memory

In [ ]:
df_small.sum(axis=0, numeric_only=True)

,0
Anxiety,3623
Bipolar,2501
Depression,15094
Normal,16040
Personality disorder,895
Stress,2296
Suicidal,10644


## Clean and Normalize Text

In [ ]:
small_statements = df_small['statement']

In [ ]:
!pip3 install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00


In [ ]:
import re
import html
import ftfy

def clean_text(text):
    # Fix encoding
    text = ftfy.fix_text(text)

    # Convert HTML entities
    text = html.unescape(text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
cleaned_small_statements = small_statements.apply(clean_text)

## Putting the statement back

In [ ]:
labels = df_small.drop(columns=['statement'])
df_small = pd.concat([cleaned_small_statements,labels], axis=1)
df_small.head(3)

,statement,Anxiety,Bipolar,Depression,Normal,Personality disorder,Stress,Suicidal
0,"It was the summer, I had just started a new jo...",0,0,0,1,0,0,0
1,She resents my relationship with our son - My ...,0,0,0,0,0,1,0
2,"He grew from a short, stubby, orange haired, f...",0,0,0,1,0,0,0


# Dataset 2

## Initialize dataset

In [ ]:
big_datafile = '/content/drive/MyDrive/Mental-health-related-subreddits.csv' # where the csv file is
df_big = pd.read_csv(big_datafile)

# remove duplicates
df_big = df_big.drop_duplicates(ignore_index=True)

# size without the duplicates
print("Big Dataset's seize without duplicates: "+ str(len(df_big)) + "\n")

# distribution of features
print("Subreddits and their count: \n"+ str(df_big['Subreddit'].value_counts()) + "\n")


Big Dataset's seize without duplicates: 487557

Subreddits and their count: 
Subreddit
depression       258239
Anxiety           86132
bipolar           41100
mentalhealth      39289
BPD               38166
schizophrenia     17491
autism             7137
Name: count, dtype: int64



## Check for null rows

In [ ]:
def show_null_count(df:pd.DataFrame):
    for column in df.columns:
        num = 0
        try:
            num = df[column].isnull().value_counts()[True]
        except:
            num = 0
        print("Number of null values for {}: {}".format(column, num))

In [ ]:
show_null_count(df_big)

Number of null values for Title: 5
Number of null values for Text: 0
Number of null values for Subreddit: 3


## Remove null subreddits

In [ ]:
# drop instances with a null subreddit
df_big = df_big.dropna(subset='Subreddit', ignore_index = True)

show_null_count(df_big)

Number of null values for Title: 2
Number of null values for Text: 0
Number of null values for Subreddit: 0


## Replace null titles

In [ ]:
# replace null values in Title with an empty string
df_big = df_big.fillna("")
show_null_count(df_big)

Number of null values for Title: 0
Number of null values for Text: 0
Number of null values for Subreddit: 0


In [ ]:
text_counts = df_big['Text'].value_counts()
text_counts.loc[text_counts >= 10]

,count
Text,
\[removed\],81
.,53
\n,46
"Greetings &amp; Salutations!\n\nUse this post to introduce yourself if you're new. Or maybe you're not so new, but haven't gotten around to introducing yourself yet in one of these posts. That's ok too! Either way, we'd love to offer you a warm welcome to our community. In fact, if you've introduced yourself before, why not take some time to say hi to the new people commenting here? What do you have going on this week that's giving you anxiety? Talk to us, we can do this together - **you're not alone in this**.\n\n--- \n\n###Question of the week:\n---\n\nWhat is something you keep meaning to do, but it just never quite happens? - For me it's ice skating. I'm crossing my fingers this year is the year! \n\n--- \n---\n\n\n**Come chat with us!** That's right we have an /r/Anxiety irc channel were we hang out and talk about random things, or help those who are having a hard time. Tons of great people so feel free to stop on in and say hello! [Chatroom Weblink](https://kiwiirc.com/client/irc.snoonet.org/anxiety) : [More Information](http://www.reddit.com/r/Anxiety/wiki/irc)\n\n*********\n\n[Wiki](https://www.reddit.com/r/Anxiety/wiki/index) | [FAQ](http://www.reddit.com/r/Anxiety/wiki/faq) | [Types of Anxiety](https://www.reddit.com/r/Anxiety/wiki/anxiety_subtypes) | [Online Resources &amp; Downloads](http://www.reddit.com/r/Anxiety/wiki/onlineresources) | [Anxiety Chatroom](https://app.orangechat.io/r/anxiety) | [Anxiety Sub Community Map](https://redd.it/5ff4bn)\n",31
:(,30
&amp;#x200B;,30
Title,25
That is all.,17
title,17


## Replace meaningless or placeholder titles and texts

In [ ]:
# the placeholders are only what I found in the dataset when sorting by value counts. It isn't exhaustive.
placeholder = ['\\[removed\\]','.','title','&amp;#x200b;','deleted','[deleted]','title.','title says it all','title says it all.','\\n']

def replace_placeholder(x):
    if x.strip().lower() in placeholder:
        return ""
    else:
        return x

text = df_big['Text'].apply(replace_placeholder)
title = df_big['Title'].apply(replace_placeholder)

df_big = pd.DataFrame({'Title': title, 'Text': text, 'Subreddit': df_big['Subreddit']})
df_big.head(3)

,Title,Text,Subreddit
0,exposure does not work!,I have struggled with social anxiety from chil...,Anxiety
1,Panic attack? derealization? can't go to docto...,"Back in March (I know, a while ago D:), I woke...",Anxiety
2,How long can a panic attack last?!,I've been withdrawing from medicines lately (e...,Anxiety


In [ ]:
import re

res = []
for row in df_big.itertuples(index=False):
    if bool(re.search(r'[.?!]$', row.Title)):
        res.append(row.Title + ' ' + row.Text)
    else:
        # if there isn't an end of sentence punctuation mark at the end of the string, add a period
        res.append(row.Title + '. ' + row.Text)
big_statements = pd.Series(data=res, name='Text')
cleaned_big_statements = big_statements.apply(clean_text)
df_big = pd.concat([cleaned_big_statements, df_big['Subreddit']], axis=1)
df_big.head(3)

,Text,Subreddit
0,exposure does not work! I have struggled with ...,Anxiety
1,Panic attack? derealization? can't go to docto...,Anxiety
2,How long can a panic attack last?! I've been w...,Anxiety


## Combine instances of the same text but with different subreddit using multilabel binarizer.

In [ ]:
df_big_multilabel = get_multilabels(df_big, 'Subreddit', 'Text')
df_big_multilabel.head(3)

,Text,Anxiety,BPD,autism,bipolar,depression,mentalhealth,schizophrenia
0,!! Had an amazing evening with my crush and my...,1,0,0,0,0,0,0
1,!!! dating another borderline. Is it a bad ide...,0,1,0,0,0,0,0
2,!!!!! https://i.redd.it/789ohk8f8ye11.png,0,0,1,0,0,0,0


In [ ]:
duplicated_text = df_big.loc[df_big.duplicated(subset='Text', keep=False)].iat[0,0]
df_big.loc[df_big['Text'] == duplicated_text]

,Text,Subreddit
30,This feeling of constant inferiority. This con...,Anxiety
124402,This feeling of constant inferiority. This con...,depression


In [ ]:
df_big_multilabel.loc[df_big_multilabel['Text'] == duplicated_text]

,Text,Anxiety,BPD,autism,bipolar,depression,mentalhealth,schizophrenia
405495,This feeling of constant inferiority. This con...,1,0,0,0,1,0,0


In [ ]:
show_null_count(df_big_multilabel)

Number of null values for Text: 0
Number of null values for Anxiety: 0
Number of null values for BPD: 0
Number of null values for autism: 0
Number of null values for bipolar: 0
Number of null values for depression: 0
Number of null values for mentalhealth: 0
Number of null values for schizophrenia: 0


In [ ]:
df_big_multilabel.sum(axis=0, numeric_only=True)

,0
Anxiety,86115
BPD,38156
autism,7136
bipolar,41087
depression,258179
mentalhealth,39270
schizophrenia,17488


In [ ]:
counts = df_big_multilabel['Text'].value_counts()
counts.loc[counts > 1]

,count
Text,


all texts are unique

In [ ]:
df_big = df_big_multilabel.copy()
df_big_multilabel = [] # remove df_big_multilabel from memory
len(df_big)

486475

# Save processed datasets



In [ ]:
path_df_small = file_path + 'df_small.pkl'
path_df_big = file_path + 'df_big.pkl'

# Save the dataframes
df_small.to_pickle(path_df_small)
df_big.to_pickle(path_df_big)